# Dry run — Model: FacebookAI/xlm-roberta-large

Quick sanity run before B's full pipeline lands: fine-tune on a **3,000-row stratified sample**, check it learns (not collapsed), measure speed, and write a valid `submission_dryrun.csv`.

**Kaggle settings:** Accelerator = GPU T4×2 (uses 1 GPU), Internet = off. Attach the competition data and the `xlm-roberta-large` model. Expected runtime ≈ 10 min.

In [ ]:
import os, glob, time
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]  # fixed label map, same as the team contract
label2id = {l: i for i, l in enumerate(LABELS)}

N_TRAIN, N_VAL = 3000, 1000
MAX_LEN, BS, LR, EPOCHS, WARMUP, SEED = 128, 16, 1e-5, 2, 0.1, 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda")
print(torch.__version__, torch.cuda.get_device_name(0))

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
MODEL_DIR = os.path.dirname([p for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
                             if "xlm-roberta-large" in p.lower()][0])
print("data:", DATA_DIR, "\nmodel:", MODEL_DIR)

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
print(pd.crosstab(train.language, train.label, margins=True))

tr, va = train_test_split(train, train_size=N_TRAIN, test_size=N_VAL, random_state=SEED,
                          stratify=train.label + "_" + train.language)
y_tr = tr.label.map(label2id).values
y_va = va.label.map(label2id).values

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
encode = lambda texts: tok(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]
X_tr, X_va, X_te = encode(tr.text), encode(va.text), encode(test.text)

lens = np.array([len(x) for x in tok(list(train.text.sample(2000, random_state=SEED)))["input_ids"]])
print("token length p50/p90/p99:", np.percentile(lens, [50, 90, 99]).astype(int),
      "| >256:", f"{(lens > 256).mean():.1%}", "| >512:", f"{(lens > 512).mean():.1%}")

def batches(X, y=None, shuffle=False):
    # shuffled for training, length-sorted for fast inference
    order = np.random.permutation(len(X)) if shuffle else np.argsort([len(x) for x in X])
    for i in range(0, len(order), BS):
        idx = order[i:i + BS]
        enc = tok.pad({"input_ids": [X[j] for j in idx]}, return_tensors="pt").to(device)
        yield idx, enc, (torch.tensor(y[idx], device=device) if y is not None else None)

@torch.no_grad()
def predict(model, X):
    model.eval()
    probs = np.zeros((len(X), 3), dtype=np.float32)
    for idx, enc, _ in batches(X):
        with torch.autocast("cuda", dtype=torch.float16):
            probs[idx] = model(**enc).logits.float().softmax(-1).cpu().numpy()
    return probs

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=3, local_files_only=True).to(device)

freq = np.bincount(y_tr, minlength=3) / len(y_tr)
w = 1 / np.sqrt(freq); w = w / w.mean()
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
print("class weights:", dict(zip(LABELS, w.round(2))))

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * int(np.ceil(len(X_tr) / BS))
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.amp.GradScaler("cuda")

t0, step = time.time(), 0
for epoch in range(EPOCHS):
    model.train()
    for idx, enc, yb in batches(X_tr, y_tr, shuffle=True):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(**enc).logits.float(), yb)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        step += 1
        if step % 50 == 0:
            print(f"ep {epoch} step {step}/{steps} loss {loss.item():.4f} | {step * BS / (time.time() - t0):.1f} samples/s")
    va_pred = predict(model, X_va).argmax(1)
    print(f"== epoch {epoch}: val macro-F1 {f1_score(y_va, va_pred, average='macro'):.4f}")

sps = steps * BS / (time.time() - t0)
# shortcut: assumes time scales ~linearly with max_len; real 512 runs are a bit slower, re-measure in B's pipeline
for L in (128, 256, 384, 512):
    print(f"est. 1 fold x 3 epochs at max_len {L}: {11200 * 3 / sps * L / MAX_LEN / 60:.0f} min")

In [ ]:
va_probs = predict(model, X_va)
va_pred = va_probs.argmax(1)
macro = f1_score(y_va, va_pred, average="macro")
print(classification_report(y_va, va_pred, target_names=LABELS, digits=4))
for lang, g in va.assign(pred=va_pred, y=y_va).groupby("language"):
    print(f"{lang}: macro-F1 {f1_score(g.y, g.pred, average='macro'):.4f}  (n={len(g)})")
print("pred distribution:", dict(zip(LABELS, np.bincount(va_pred, minlength=3))))
if macro < 0.40:
    print("WARNING: looks collapsed (macro-F1 < 0.40) — rerun with another SEED before trusting the setup")

In [ ]:
te_pred = predict(model, X_te).argmax(1)
sub = pd.DataFrame({"id": test.id, "label": [LABELS[i] for i in te_pred]})

assert list(sub.columns) == ["id", "label"] and len(sub) == len(test) == 6000
assert sub.id.is_unique and set(sub.id) == set(test.id)
assert sub.label.isin(LABELS).all()
sub.to_csv("submission_dryrun.csv", index=False)
print(sub.label.value_counts(normalize=True).round(3))
print("wrote submission_dryrun.csv — hand it to A; only A submits")